# Notebook 6 — EEG Conformer: A CNN-Transformer Hybrid for EEG Decoding

---

## Section 0 — What we're building on

### Prior concepts you already have

From the **EEGNet notebook**, you know:
- **Depthwise convolution** acts as a learned spatial filter — each filter sees exactly one input channel. With `groups=F1`, the depthwise conv learns `D` spatial filters per temporal filter, collapsing the channel dimension from `C=22` to `1`.
- **Separable convolution** (depthwise + pointwise) captures the same expressiveness as standard conv with far fewer parameters.
- EEGNet's architecture maps directly onto EEG signal structure: temporal conv → spatial conv → separable temporal conv → classifier.
- EEGNet achieves a certain accuracy/kappa across 9 subjects. These are your deep learning baseline numbers.

From your **Transformer experience** (GPT/ViT projects), you know:
- Scaled dot-product attention: $\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$
- Multi-head attention splits the embedding into `h` heads, applies attention independently, and concatenates.
- Layer normalization, residual connections, and position-wise feed-forward networks.
- Positional encoding — the Transformer has no built-in notion of sequence order.

From **CSP + LDA/SVM**, you know:
- CSP uses fixed spatial filters and log-variance features — a strong baseline on small datasets.
- Your CSP numbers across 9 subjects serve as the traditional ML benchmark.

### What this notebook adds

The **EEG Conformer** (Song et al., 2023) is a hybrid CNN-Transformer that addresses a fundamental limitation of pure CNN architectures like EEGNet: **fixed receptive fields**.

EEGNet's temporal kernels see a fixed window of time (e.g. 125 samples = 0.5s). If two discriminative EEG events are separated by more than this window — for instance, an initial ERD onset at 0.2s and a sustained pattern at 0.8s — EEGNet cannot directly model the relationship between them. Each conv layer only sees its local neighborhood.

Self-attention has no such limitation: every position can attend to every other position in a single layer. This is why Transformers excel at capturing **long-range temporal dependencies**.

The Conformer combines both:
1. A **CNN front-end** (similar to EEGNet's Block 1) that extracts local spatial-temporal features and produces a sequence of patch embeddings.
2. A **Transformer encoder** that models global temporal relationships between these patches via self-attention.
3. A **classification head** that predicts the motor imagery class from the enriched representations.

### Why this matters for BCI

Motor imagery signals are not just "a burst of ERD." The full temporal dynamics include preparation (readiness potential), ERD onset, sustained desynchronization, and post-movement ERS (event-related synchronization). Capturing the temporal relationship between these phases — not just detecting each one individually — is what gives Conformer its edge on longer epochs. On our short 1-second windows, the advantage may be smaller, and this is something we'll quantify.

> **Why "Conformer"?** The name comes from "Convolutional Transformer" — the same portmanteau used in speech recognition (Gulati et al., 2020). The EEG version (Song et al., 2023) adapts this idea specifically for EEG signals. It is also referred to as "EEG Conformer" in the literature.


---

## Part 1 — Setup and Imports


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay
import os
import sys

# Import preprocessing pipeline
sys.path.insert(0, '../4.data preprocessing')
from run_pipeline import run_pipeline

DATA_DIR = '../mne_data/bci_iv_2a'

In [ ]:
# Device configuration
import torch

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print(f"Using device: {DEVICE}")

In [ ]:
# ⚠️  COMPUTE NOTE
# EEG Conformer has more parameters than EEGNet and trains slower.
# On MPS (Apple Silicon): ~40–60 min for all 9 subjects.
# On CPU: several hours. NOT recommended.
# Recommended: Google Colab with T4 GPU.
# Set Runtime → Change runtime type → GPU in Colab.

### Load data for one subject (development)

We'll develop on Subject A01, then sweep all 9 subjects at the end.

In [ ]:
data = run_pipeline('A01', DATA_DIR)
X_train = data['X_train']
X_test  = data['X_test']
y_train = data['y_train']
y_test  = data['y_test']

print(f"X_train: {X_train.shape}")  # (n_windows, 22, 250)
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}, classes: {np.unique(y_train)}")
print(f"y_test:  {y_test.shape},  classes: {np.unique(y_test)}")

---

## Part 2 — PyTorch Dataset

Same as EEGNet — wrap NumPy arrays and add the singleton dimension.

**Input format:** `(B, 1, C, T)` where `B`=batch, `1`=singleton kernel dim, `C`=22 channels, `T`=250 timepoints.

You already implemented this in EEGNet. Re-implement it here from scratch (same logic, same shape contract).

In [ ]:
class EEGDataset(Dataset):
    """
    Wraps EEG data for PyTorch DataLoader.
    
    Args:
        X: np.ndarray, shape (N, C, T) — preprocessed EEG windows
        y: np.ndarray, shape (N,) — integer class labels 0–3
    
    __getitem__ returns:
        x: FloatTensor, shape (1, C, T) — one trial with singleton dim
        label: LongTensor, scalar
    """
    def __init__(self, X, y):
        # YOUR CODE HERE
        raise NotImplementedError
    
    def __len__(self):
        # YOUR CODE HERE
        raise NotImplementedError
    
    def __getitem__(self, idx):
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
# Sanity check — Dataset
_ds = EEGDataset(X_train, y_train)
_x, _y = _ds[0]
assert _x.shape == (1, 22, 250), f"Expected (1,22,250), got {_x.shape}"
assert _x.dtype == torch.float32
assert _y.dtype == torch.long
print(f"✓ Dataset: {len(_ds)} samples, sample shape {_x.shape}")

### Create DataLoaders

In [ ]:
BATCH_SIZE = 64

train_ds = EEGDataset(X_train, y_train)
test_ds  = EEGDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f"Train: {len(train_loader)} batches | Test: {len(test_loader)} batches")

---

## Part 3 — Patch Embedding: From EEG to Token Sequences

### The key idea: tokenizing a continuous signal

Transformers operate on **sequences of tokens**. In NLP, each token is a word or subword. In ViT (Vision Transformer), each token is a flattened image patch. For EEG, we need to convert a continuous multi-channel time series into a sequence of discrete tokens.

The EEG Conformer does this in two steps:

**Step 1 — Temporal convolution:** A 1D conv across the time axis (same as EEGNet's first layer) extracts local temporal features. This produces `F1` feature maps of shape `(F1, C, T)` from the input `(1, C, T)`.

**Step 2 — Spatial convolution (depthwise):** A depthwise conv across channels collapses `C=22` into `1`, producing shape `(F1*D, 1, T)` → squeeze to `(F1*D, T)`.

At this point, we have a 1D feature sequence of length `T` with `F1*D` features per timepoint. But `T=250` is too long for self-attention (quadratic cost). We need to **compress** the time axis.

**Step 3 — Temporal pooling as patch creation:** An average pooling with kernel `pool_size` and stride `pool_stride` reduces the time axis from `T` to `T' = T // pool_stride` (with appropriate pooling). Each pooled segment corresponds to one temporal "patch" — a compressed representation of a short time window.

The result is a sequence of `T'` tokens, each of dimension `d_model = F1 * D`. This is the Transformer's input.

### Dimension walkthrough

Let's trace the shapes with default parameters (`F1=40`, `D=1`, `kern_len=25`, `pool_size=75`, `pool_stride=15`):

```
Input:          (B, 1, 22, 250)           — raw EEG with singleton dim
                     │
Temporal conv:  (B, 40, 22, 250)          — F1=40 temporal filters, kernel=(1, 25)
                     │
BatchNorm + ELU
                     │
Depthwise conv: (B, 40, 1, 250)           — D=1, each filter sees one channel, collapse C→1
                     │
BatchNorm + ELU
                     │
Squeeze dim 2:  (B, 40, 250)              — remove the spatial dim (now 1D signal)
                     │
AvgPool1d:      (B, 40, T')               — T' depends on pool_size and pool_stride
                     │                       T' = floor((250 - 75) / 15) + 1 = 12
Transpose:      (B, T', 40)               — (batch, seq_len, d_model) — Transformer format
```

The Transformer now sees a sequence of **12 tokens**, each a 40-dimensional embedding. Each token represents a ~60ms window of EEG (15 samples / 250 Hz), compressed from 22 channels into 40 features.

> **Why a CNN front-end, not raw patch projection?** ViT uses a simple linear projection to embed image patches. That works because image patches are spatially local and dense with information. EEG is different: the signal is noisy, multi-channel, and the discriminative features (ERD in specific frequency bands at specific channels) need to be extracted before the Transformer can attend to them. The CNN front-end acts as a feature extractor — it handles the spatial and temporal filtering that the Transformer is not well-suited for. This is the "Conformer" insight: let CNNs do local feature extraction, let Transformers do global temporal reasoning.

> **Why not use the exact EEGNet architecture as the front-end?** You could, but the original EEG Conformer paper uses a simpler two-layer CNN (temporal + depthwise) without the separable conv block. The separable block in EEGNet further compresses the time axis — we want to preserve enough temporal resolution for the Transformer to have a meaningful sequence to attend over. If the CNN compresses too aggressively, the Transformer has nothing to do.

### Hyperparameter differences from EEGNet

Notice that the Conformer uses **different** CNN hyperparameters than EEGNet:

| Parameter | EEGNet | Conformer | Why different? |
|-----------|--------|-----------|----------------|
| F1 (temporal filters) | 8 | 40 | Conformer needs richer patch embeddings for attention |
| D (depth multiplier) | 2 | 1 | Kept simple — Transformer handles feature mixing |
| kern_len (temporal kernel) | 125 | 25 | Shorter — Transformer handles long-range deps |
| d_model | — | 40 (=F1×D) | Transformer embedding dimension |

The key tradeoff: EEGNet needs a long temporal kernel (125 = 0.5s) because it has no mechanism for long-range dependencies. The Conformer can use a short kernel (25 = 0.1s) because the Transformer handles global context.

### Task 3.1 — Implement PatchEmbedding

Implement the CNN front-end that converts raw EEG `(B, 1, C, T)` into a sequence of patch tokens `(B, T', d_model)`.

**Layers (in order):**
1. `nn.Conv2d(1, F1, (1, kern_len), padding=(0, kern_len//2), bias=False)` — temporal conv
2. `nn.BatchNorm2d(F1)`
3. `nn.ELU()`
4. `nn.Conv2d(F1, F1*D, (n_channels, 1), groups=F1, bias=False)` — depthwise spatial conv
5. `nn.BatchNorm2d(F1*D)`
6. `nn.ELU()`
7. `nn.Dropout(dropout)`
8. `nn.AvgPool1d(pool_size, stride=pool_stride)` — temporal compression (applied after squeezing dim 2)

**Forward pass:**
1. Pass through temporal conv → BN → ELU → depthwise conv → BN → ELU → dropout: output is `(B, F1*D, 1, T)`.
2. Squeeze spatial dim: `(B, F1*D, 1, T)` → `(B, F1*D, T)` via `.squeeze(2)`.
3. Apply `AvgPool1d`: `(B, F1*D, T)` → `(B, F1*D, T')`.
4. Transpose to Transformer format: `(B, F1*D, T')` → `(B, T', F1*D)` via `.transpose(1, 2)`.

**Return:** `(B, T', d_model)` where `d_model = F1 * D`.

In [ ]:
class PatchEmbedding(nn.Module):
    """
    CNN front-end that converts raw EEG into a sequence of patch tokens.
    
    Args:
        n_channels: int, number of EEG channels (22)
        F1: int, number of temporal filters (40)
        D: int, depth multiplier (1)
        kern_len: int, temporal kernel size (25)
        pool_size: int, average pooling kernel (75)
        pool_stride: int, pooling stride (15)
        dropout: float, dropout rate (0.5)
    
    forward(x):
        x: (B, 1, C, T)
        returns: (B, T', d_model) where d_model = F1 * D
    """
    def __init__(self, n_channels=22, F1=40, D=1, kern_len=25,
                 pool_size=75, pool_stride=15, dropout=0.5):
        super().__init__()
        self.d_model = F1 * D
        
        # ============================================================
        # YOUR CODE HERE: define the CNN layers and pooling
        # ============================================================
        raise NotImplementedError
    
    def forward(self, x):
        """
        Args:
            x: (B, 1, C, T) — e.g. (B, 1, 22, 250)
        Returns:
            tokens: (B, T', d_model) — e.g. (B, 12, 40)
        """
        # ============================================================
        # YOUR CODE HERE:
        # 1. CNN layers: (B,1,C,T) → (B, F1*D, 1, T)
        # 2. Squeeze dim 2: → (B, F1*D, T)
        # 3. AvgPool1d: → (B, F1*D, T')
        # 4. Transpose: → (B, T', F1*D)
        # ============================================================
        raise NotImplementedError

In [ ]:
# Sanity check — PatchEmbedding
_pe = PatchEmbedding().to(DEVICE)
with torch.no_grad():
    _x = torch.randn(4, 1, 22, 250).to(DEVICE)
    _tokens = _pe(_x)

# With pool_size=75, pool_stride=15: T' = floor((250 - 75) / 15) + 1 = 12
_expected_T = (250 - 75) // 15 + 1  # = 12
assert _tokens.shape == (4, _expected_T, 40), f"Expected (4, {_expected_T}, 40), got {_tokens.shape}"
assert _tokens.dtype == torch.float32
print(f"✓ PatchEmbedding: (4, 1, 22, 250) → {_tokens.shape}")
print(f"  Sequence length T' = {_tokens.shape[1]} tokens, each of dim d_model = {_tokens.shape[2]}")
print(f"  Each token represents {75/250*1000:.0f}ms of EEG, strided by {15/250*1000:.0f}ms")

---

## Part 4 — Positional Encoding

### Why position matters

Self-attention is permutation-equivariant — if you shuffle the order of tokens, the output changes only because of the positional encoding. Without it, the Transformer treats the token sequence as an unordered set. For EEG, temporal order is critical: an ERD event at 200ms means something different than one at 800ms.

### Learnable vs sinusoidal

You've seen sinusoidal positional encoding from the original Transformer (Vaswani et al.). For EEG, we use **learnable positional embeddings** — a simple `nn.Parameter` of shape `(1, T', d_model)` added to the token sequence. This is simpler and works well when the sequence length is fixed (which it is for us — always `T'` tokens).

The implementation is straightforward:
```python
self.pos_embed = nn.Parameter(torch.zeros(1, max_len, d_model))
```

In `forward`, just add it: `x = x + self.pos_embed[:, :seq_len, :]`

We initialize with zeros and let the optimizer learn the positions — this is the standard approach from ViT.

> **Why learnable?** With only 12 tokens, there aren't enough positions for the structured sinusoidal encoding to offer any advantage. Learnable embeddings are simpler and give the model full flexibility. For very long sequences (hundreds of tokens), sinusoidal encoding generalizes better to unseen lengths — but that's not our case.

---

## Part 5 — Transformer Encoder Block

### Architecture of one Transformer encoder layer

You already know the components. Here's how they wire together:

```
Input: (B, T', d_model)
       │
  ┌────▼────┐
  │ LayerNorm │
  └────┬────┘
       │
  ┌────▼──────────────┐
  │ Multi-Head Attn    │
  │ (h heads, d_k=d/h) │
  └────┬──────────────┘
       │
  ┌────▼────┐
  │ Dropout  │
  └────┬────┘
       ⊕ ← residual
       │
  ┌────▼────┐
  │ LayerNorm │
  └────┬────┘
       │
  ┌────▼────────────────────────┐
  │ Feed-Forward Network         │
  │ Linear(d, d*ff_ratio)        │
  │ → GELU → Dropout             │
  │ → Linear(d*ff_ratio, d)      │
  │ → Dropout                    │
  └────┬────────────────────────┘
       ⊕ ← residual
       │
Output: (B, T', d_model)
```

This is **Pre-LN** (layer norm before attention/FFN), which is the modern default — it makes training more stable than the original Post-LN from Vaswani et al.

### Using `nn.MultiheadAttention`

PyTorch provides `nn.MultiheadAttention(embed_dim, num_heads, dropout, batch_first=True)`. 

**Key argument:** `batch_first=True` means input/output shapes are `(B, T', d_model)` — matching our convention. Without this flag, PyTorch expects `(T', B, d_model)`.

**Calling convention:** `attn_out, attn_weights = self.attn(query, key, value)`. For self-attention, all three are the same tensor.

**Return:** `attn_out` is `(B, T', d_model)`. `attn_weights` is `(B, T', T')` — the attention matrix. We'll use `attn_weights` later for visualization.

### Task 5.1 — Implement TransformerBlock

**Layers:**
1. `nn.LayerNorm(d_model)` — pre-attention norm
2. `nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)` — self-attention
3. `nn.Dropout(dropout)` — post-attention dropout
4. `nn.LayerNorm(d_model)` — pre-FFN norm
5. FFN: `Linear(d_model, d_model * ff_ratio)` → `GELU` → `Dropout` → `Linear(d_model * ff_ratio, d_model)` → `Dropout`

**Forward:**
1. `residual = x`
2. `x = LayerNorm1(x)` → `x, attn_weights = MultiheadAttention(x, x, x)` → `Dropout` → `x = x + residual`
3. `residual = x`
4. `x = LayerNorm2(x)` → `x = FFN(x)` → `x = x + residual`
5. Return `x` and `attn_weights`

In [ ]:
class TransformerBlock(nn.Module):
    """
    Single Transformer encoder layer with Pre-LN.
    
    Args:
        d_model: int, embedding dimension (40)
        n_heads: int, number of attention heads (10)
        ff_ratio: int, FFN expansion ratio (3)
        dropout: float, dropout rate (0.5)
    
    forward(x):
        x: (B, T', d_model)
        returns: (output, attn_weights)
            output: (B, T', d_model)
            attn_weights: (B, T', T')
    """
    def __init__(self, d_model=40, n_heads=10, ff_ratio=3, dropout=0.5):
        super().__init__()
        
        # ============================================================
        # YOUR CODE HERE: define LayerNorms, MultiheadAttention, FFN
        # ============================================================
        raise NotImplementedError
    
    def forward(self, x):
        """
        Args:
            x: (B, T', d_model)
        Returns:
            out: (B, T', d_model)
            attn_weights: (B, T', T') — attention weights for visualization
        """
        # ============================================================
        # YOUR CODE HERE:
        # Pre-LN attention block with residual
        # Pre-LN FFN block with residual
        # Return both output and attention weights
        # ============================================================
        raise NotImplementedError

In [ ]:
# Sanity check — TransformerBlock
_tb = TransformerBlock(d_model=40, n_heads=10).to(DEVICE)
with torch.no_grad():
    _tokens = torch.randn(4, 12, 40).to(DEVICE)
    _out, _attn = _tb(_tokens)

assert _out.shape == (4, 12, 40), f"Expected (4,12,40), got {_out.shape}"
assert _attn.shape == (4, 12, 12), f"Expected (4,12,12), got {_attn.shape}"

# Attention weights should sum to ~1 along the last dim (softmax output)
_attn_sum = _attn.sum(dim=-1)
assert torch.allclose(_attn_sum, torch.ones_like(_attn_sum), atol=1e-4), \
    f"Attention weights should sum to 1, got {_attn_sum[0,0]:.4f}"

print(f"✓ TransformerBlock: (4, 12, 40) → {_out.shape}, attn: {_attn.shape}")
print(f"  Attention row sums ≈ 1.0: {_attn_sum[0,0].item():.4f}")

---

## Part 6 — The Full EEG Conformer

### Architecture overview

Now we assemble the pieces:

```
Input: (B, 1, 22, 250)
        │
  ┌─────▼──────────────────────────┐
  │ PatchEmbedding (CNN front-end)  │  → (B, T', d_model)
  └─────┬──────────────────────────┘
        │
  ┌─────▼──────────────────────────┐
  │ + Learnable positional embed    │  → (B, T', d_model)
  └─────┬──────────────────────────┘
        │
  ┌─────▼──────────────────────────┐
  │ TransformerBlock × n_layers     │  → (B, T', d_model)
  └─────┬──────────────────────────┘
        │
  ┌─────▼──────────────────────────┐
  │ Classification head             │  → (B, n_classes)
  │ LayerNorm → mean pool → Linear  │
  └────────────────────────────────┘
```

### Classification strategy: mean pooling

In NLP Transformers, you typically use the `[CLS]` token for classification. For EEG, we use **mean pooling** over the sequence dimension: average all `T'` token representations into one vector, then classify with a linear layer. This is simpler and works well because every temporal patch carries relevant motor imagery information (unlike NLP where some tokens are filler).

$$\mathbf{h} = \frac{1}{T'} \sum_{t=1}^{T'} \mathbf{z}_t \quad \text{where } \mathbf{z}_t \in \mathbb{R}^{d_{\text{model}}}$$

Then: $\text{logits} = \mathbf{W}_c \mathbf{h} + \mathbf{b}_c$, where $\mathbf{W}_c \in \mathbb{R}^{n_{\text{classes}} \times d_{\text{model}}}$.

### Default hyperparameters

| Parameter | Value | Meaning |
|-----------|-------|---------|
| F1 | 40 | Temporal filters in CNN |
| D | 1 | Depth multiplier |
| kern_len | 25 | Temporal kernel (~100ms at 250Hz) |
| pool_size | 75 | Pooling window (~300ms) |
| pool_stride | 15 | Pooling stride (~60ms) |
| d_model | 40 (=F1×D) | Transformer embedding dim |
| n_heads | 10 | Attention heads (d_k = 40/10 = 4 per head) |
| n_layers | 6 | Transformer encoder layers |
| ff_ratio | 3 | FFN expansion (40 → 120 → 40) |
| dropout | 0.5 | Same as EEGNet — heavy regularization for small data |

> **Why 10 heads with d_model=40?** Each head has d_k = 4, which is small but sufficient for a 12-token sequence. More heads = more diverse attention patterns. With only 12 tokens, even small d_k can capture the relevant pairwise relationships.

### Task 6.1 — Implement EEGConformer

In [ ]:
class EEGConformer(nn.Module):
    """
    EEG Conformer: CNN front-end + Transformer encoder + classifier.
    
    Args:
        n_channels: int (22)
        n_timepoints: int (250)
        n_classes: int (4)
        F1: int, temporal filters (40)
        D: int, depth multiplier (1)
        kern_len: int, temporal kernel (25)
        pool_size: int, pooling window (75)
        pool_stride: int, pooling stride (15)
        n_heads: int, attention heads (10)
        n_layers: int, Transformer layers (6)
        ff_ratio: int, FFN expansion (3)
        dropout: float (0.5)
    
    forward(x):
        x: (B, 1, C, T)
        returns: logits (B, n_classes)
    """
    def __init__(self, n_channels=22, n_timepoints=250, n_classes=4,
                 F1=40, D=1, kern_len=25,
                 pool_size=75, pool_stride=15,
                 n_heads=10, n_layers=6, ff_ratio=3, dropout=0.5):
        super().__init__()
        
        d_model = F1 * D
        self.d_model = d_model
        self.n_layers = n_layers
        
        # Compute sequence length after pooling
        self.seq_len = (n_timepoints - pool_size) // pool_stride + 1
        
        # ============================================================
        # YOUR CODE HERE:
        # 1. self.patch_embed = PatchEmbedding(...)
        # 2. self.pos_embed  = nn.Parameter(torch.zeros(1, self.seq_len, d_model))
        # 3. self.pos_drop   = nn.Dropout(dropout)
        # 4. self.blocks     = nn.ModuleList([TransformerBlock(...) for _ in range(n_layers)])
        # 5. self.norm       = nn.LayerNorm(d_model)   — final norm before classifier
        # 6. self.classifier = nn.Linear(d_model, n_classes)
        # ============================================================
        raise NotImplementedError
    
    def forward(self, x):
        """
        Args:
            x: (B, 1, C, T)
        Returns:
            logits: (B, n_classes)
        """
        # ============================================================
        # YOUR CODE HERE:
        # 1. tokens = self.patch_embed(x)          → (B, T', d_model)
        # 2. tokens = tokens + self.pos_embed      → add positional
        # 3. tokens = self.pos_drop(tokens)
        # 4. for block in self.blocks:
        #        tokens, _ = block(tokens)          → pass through each layer
        # 5. tokens = self.norm(tokens)             → final layer norm
        # 6. pooled = tokens.mean(dim=1)            → mean pool over T'
        # 7. logits = self.classifier(pooled)       → (B, n_classes)
        # ============================================================
        raise NotImplementedError
    
    def get_attention_weights(self, x):
        """
        Run forward pass and collect attention weights from all layers.
        Used for visualization — not for training.
        
        Args:
            x: (B, 1, C, T)
        Returns:
            attn_list: list of (B, T', T') tensors, one per layer
        """
        self.eval()
        with torch.no_grad():
            tokens = self.patch_embed(x)
            tokens = tokens + self.pos_embed
            tokens = self.pos_drop(tokens)
            
            attn_list = []
            for block in self.blocks:
                tokens, attn_w = block(tokens)
                attn_list.append(attn_w.cpu())
        
        return attn_list

In [ ]:
# Sanity check — EEGConformer
_model = EEGConformer().to(DEVICE)

with torch.no_grad():
    _x = torch.randn(4, 1, 22, 250).to(DEVICE)
    _out = _model(_x)

assert _out.shape == (4, 4), f"Expected (4,4), got {_out.shape}"

# At random init, loss ≈ log(4) ≈ 1.386 for 4-class classification
_loss = F.cross_entropy(_out, torch.zeros(4, dtype=torch.long).to(DEVICE))
assert abs(_loss.item() - 1.386) < 0.3, f"Loss at init should be ≈log(4), got {_loss.item():.3f}"

# Check attention weights
_attn_list = _model.get_attention_weights(_x)
assert len(_attn_list) == 6, f"Expected 6 layers, got {len(_attn_list)}"
assert _attn_list[0].shape == (4, 12, 12), f"Attn shape wrong: {_attn_list[0].shape}"

# Parameter count
_n_params = sum(p.numel() for p in _model.parameters())
print(f"✓ EEGConformer: (4,1,22,250) → {_out.shape}")
print(f"  Loss at random init: {_loss.item():.3f} (expected ≈1.386)")
print(f"  Attention weights: {len(_attn_list)} layers × {_attn_list[0].shape}")
print(f"  Total parameters: {_n_params:,}")

### Decision — EEG Conformer architecture choices

1. The CNN front-end uses `kern_len=25` (100ms), compared to EEGNet's `kern_len=125` (500ms). Why can the Conformer get away with a much shorter temporal kernel? What takes over the responsibility of capturing long-range patterns?

2. The model uses `pool_size=75` and `pool_stride=15`, resulting in 12 tokens. If you changed `pool_stride=5` (giving ~36 tokens), what would happen to (a) self-attention's ability to model fine-grained temporal patterns, (b) computational cost, and (c) overfitting risk given only ~288 training trials?

3. Compare the parameter count you just printed to EEGNet's (~2,600 parameters). The Conformer is larger. Given that both models are trained on ~288 trials per subject, what regularization mechanisms prevent the larger model from simply memorizing the training data?

*Write your answers here.*

---

## Part 7 — Training Loop

Same training functions as EEGNet — `train_one_epoch` and `evaluate`. Re-implement from scratch.

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    """
    Train the model for one epoch.
    
    Args:
        model: nn.Module
        loader: DataLoader
        optimizer: torch.optim.Optimizer
        device: torch.device
    
    Returns:
        avg_loss: float — mean cross-entropy loss over all batches
    """
    # YOUR CODE HERE
    raise NotImplementedError


def evaluate(model, loader, device):
    """
    Evaluate the model on a dataset.
    
    Args:
        model: nn.Module
        loader: DataLoader
        device: torch.device
    
    Returns:
        accuracy: float
        kappa: float — Cohen's kappa
        all_preds: np.ndarray — predicted labels
        all_labels: np.ndarray — true labels
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# Sanity check — train and evaluate
_model = EEGConformer().to(DEVICE)
_opt = torch.optim.Adam(_model.parameters(), lr=1e-3)
_loss1 = train_one_epoch(_model, train_loader, _opt, DEVICE)
_loss2 = train_one_epoch(_model, train_loader, _opt, DEVICE)
assert _loss2 < _loss1 * 1.5, "Loss should generally decrease after 2 epochs"

_acc, _kappa, _preds, _labels = evaluate(_model, test_loader, DEVICE)
assert 0.0 <= _acc <= 1.0
assert len(_preds) == len(y_test)
print(f"✓ After 2 epochs: loss {_loss1:.3f} → {_loss2:.3f}, acc={_acc:.3f}, kappa={_kappa:.3f}")

del _model, _opt  # clean up

---

## Part 8 — Train on Subject A01 (Development Run)

### Training configuration

The Conformer benefits from a **lower learning rate** and a **learning rate scheduler** compared to EEGNet, because it has more parameters and the Transformer components can be sensitive to large gradients early in training.

We use:
- **AdamW** (Adam with decoupled weight decay) — the standard optimizer for Transformers.
- **Cosine annealing** scheduler — the learning rate starts at `LR` and decays smoothly to near zero over `N_EPOCHS`. This gives the Transformer time to learn attention patterns gradually.

> **Why AdamW instead of Adam?** In standard Adam, weight decay is entangled with the adaptive learning rate, which can reduce its effectiveness. AdamW applies weight decay directly to the weights, independent of the gradient statistics. This is the optimizer used in GPT, ViT, and most modern Transformer training. For our small EEG dataset, the decoupled weight decay provides a modest additional regularization effect.

```python
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
```

After each epoch: `scheduler.step()`

In [ ]:
# Training hyperparameters
N_EPOCHS = 300
LR = 5e-4
BATCH_SIZE = 64
WEIGHT_DECAY = 0.01

In [ ]:
# Train on Subject A01 with caching
METRICS_PATH_A01 = 'metrics/conformer_A01_metrics.pt'
MODEL_PATH_A01   = 'models/conformer_A01.pt'

if os.path.exists(METRICS_PATH_A01) and os.path.exists(MODEL_PATH_A01):
    a01_metrics = torch.load(METRICS_PATH_A01)
    model_a01 = EEGConformer().to(DEVICE)
    model_a01.load_state_dict(torch.load(MODEL_PATH_A01, map_location=DEVICE))
    print("Loaded cached A01 results.")
else:
    model_a01 = EEGConformer().to(DEVICE)
    optimizer = torch.optim.AdamW(model_a01.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    
    train_losses = []
    test_accs = []
    test_kappas = []
    
    for epoch in range(N_EPOCHS):
        loss = train_one_epoch(model_a01, train_loader, optimizer, DEVICE)
        scheduler.step()
        train_losses.append(loss)
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            acc, kappa, _, _ = evaluate(model_a01, test_loader, DEVICE)
            test_accs.append(acc)
            test_kappas.append(kappa)
            if (epoch + 1) % 50 == 0:
                print(f"Epoch {epoch+1}/{N_EPOCHS} | loss: {loss:.4f} | "
                      f"acc: {acc:.4f} | kappa: {kappa:.4f} | "
                      f"lr: {scheduler.get_last_lr()[0]:.6f}")
    
    # Final evaluation
    final_acc, final_kappa, final_preds, final_labels = evaluate(model_a01, test_loader, DEVICE)
    print(f"\nFinal A01: acc={final_acc:.4f}, kappa={final_kappa:.4f}")
    
    a01_metrics = {
        'train_losses': train_losses,
        'test_accs': test_accs,
        'test_kappas': test_kappas,
        'final_accuracy': final_acc,
        'final_kappa': final_kappa,
        'final_preds': final_preds,
        'final_labels': final_labels
    }
    
    os.makedirs('metrics', exist_ok=True)
    os.makedirs('models', exist_ok=True)
    torch.save(a01_metrics, METRICS_PATH_A01)
    torch.save(model_a01.state_dict(), MODEL_PATH_A01)
    print("Training complete. Saved.")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(a01_metrics['train_losses'])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

eval_epochs = list(range(1, len(a01_metrics['test_accs'])*10 + 1, 10))
if len(eval_epochs) > len(a01_metrics['test_accs']):
    eval_epochs = eval_epochs[:len(a01_metrics['test_accs'])]

axes[1].plot(eval_epochs, a01_metrics['test_accs'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Test Accuracy')
axes[1].axhline(y=0.25, color='r', linestyle='--', label='Chance (25%)')
axes[1].legend()

axes[2].plot(eval_epochs, a01_metrics['test_kappas'])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Kappa')
axes[2].set_title('Cohen\'s Kappa')

plt.suptitle('EEG Conformer — Subject A01 Training', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for A01
_preds = a01_metrics['final_preds']
_labels = a01_metrics['final_labels']
cm = confusion_matrix(_labels, _preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['Left', 'Right', 'Feet', 'Tongue'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues')
ax.set_title(f'EEG Conformer — A01 (acc={a01_metrics["final_accuracy"]:.3f})')
plt.tight_layout()
plt.show()

### Decision — Single-subject results

1. Compare the Conformer's A01 accuracy to your EEGNet and CSP+LDA numbers on the same subject. Which model performs best, and does the added complexity of the Transformer seem justified for a single subject with ~288 training trials?

2. Look at the training loss curve. Does it converge smoothly, or do you see instabilities? If there are sharp loss spikes, what might cause them? (Hint: think about the interaction between self-attention and small batch sizes.)

3. The cosine annealing scheduler reduces the LR from 5e-4 to near zero. Looking at the accuracy curve, when does accuracy plateau? Does the model continue to improve in the later epochs (low LR), or does most learning happen early?

*Write your answers here.*

---

## Part 9 — Full 9-Subject Evaluation

### Why all 9 subjects matter

Same rationale as EEGNet: single-subject results are not publishable. The mean ± std across all 9 subjects tells you whether the model generalizes or if your A01 result is an outlier.

In [ ]:
METRICS_PATH_ALL = 'metrics/conformer_all_subjects.pt'

if os.path.exists(METRICS_PATH_ALL):
    all_results = torch.load(METRICS_PATH_ALL)
    print("Loaded cached 9-subject results.")
    for subj in sorted(all_results.keys()):
        r = all_results[subj]
        print(f"  {subj}: acc={r['accuracy']:.4f}, kappa={r['kappa']:.4f}")
else:
    all_results = {}
    subjects = [f'A0{i}' for i in range(1, 10)]
    
    for subj in subjects:
        print(f"\n{'='*50}")
        print(f"Training Subject {subj}")
        print(f"{'='*50}")
        
        # Load data
        data = run_pipeline(subj, DATA_DIR)
        X_tr = data['X_train']
        X_te = data['X_test']
        y_tr = data['y_train']
        y_te = data['y_test']
        
        # Create loaders
        tr_loader = DataLoader(EEGDataset(X_tr, y_tr), batch_size=BATCH_SIZE,
                               shuffle=True, drop_last=False)
        te_loader = DataLoader(EEGDataset(X_te, y_te), batch_size=BATCH_SIZE,
                               shuffle=False, drop_last=False)
        
        # Train
        model = EEGConformer().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
        
        for epoch in range(N_EPOCHS):
            loss = train_one_epoch(model, tr_loader, optimizer, DEVICE)
            scheduler.step()
            if (epoch + 1) % 100 == 0:
                acc, kappa, _, _ = evaluate(model, te_loader, DEVICE)
                print(f"  Epoch {epoch+1}/{N_EPOCHS} | loss: {loss:.4f} | "
                      f"acc: {acc:.4f} | kappa: {kappa:.4f}")
        
        # Final eval
        acc, kappa, preds, labels = evaluate(model, te_loader, DEVICE)
        all_results[subj] = {'accuracy': acc, 'kappa': kappa}
        print(f"  Final: acc={acc:.4f}, kappa={kappa:.4f}")
        
        # Save per-subject model
        os.makedirs('models', exist_ok=True)
        torch.save(model.state_dict(), f'models/conformer_{subj}.pt')
    
    os.makedirs('metrics', exist_ok=True)
    torch.save(all_results, METRICS_PATH_ALL)
    print("\nAll subjects trained and saved.")

In [ ]:
# Summary statistics
accs   = [all_results[s]['accuracy'] for s in sorted(all_results.keys())]
kappas = [all_results[s]['kappa']    for s in sorted(all_results.keys())]

print(f"EEG Conformer — 9-Subject Results")
print(f"{'='*40}")
print(f"Accuracy : {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"Kappa    : {np.mean(kappas):.3f} ± {np.std(kappas):.3f}")
print(f"{'='*40}")
print(f"Best subject:  {sorted(all_results.keys())[np.argmax(accs)]} ({max(accs):.3f})")
print(f"Worst subject: {sorted(all_results.keys())[np.argmin(accs)]} ({min(accs):.3f})")

In [ ]:
# Per-subject bar chart
subjects = sorted(all_results.keys())
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(subjects))
bars = ax.bar(x, accs, color='steelblue', alpha=0.8)
ax.axhline(y=0.25, color='r', linestyle='--', label='Chance (25%)', alpha=0.7)
ax.axhline(y=np.mean(accs), color='green', linestyle='--',
           label=f'Mean ({np.mean(accs):.3f})', alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(subjects)
ax.set_ylabel('Accuracy')
ax.set_title('EEG Conformer — Per-Subject Accuracy')
ax.legend()
ax.set_ylim(0, 1)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{acc:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

### Decision — Cross-subject variability

1. Compare the Conformer's mean ± std accuracy to your EEGNet numbers (and CSP if you have them). Does the Conformer consistently outperform EEGNet, or only on certain subjects? Which subjects see the biggest improvement and which see degradation?

2. The Conformer has significantly more parameters than EEGNet. For subjects where the Conformer performs *worse* than EEGNet, what might be happening? (Hint: consider the size of the training set relative to the model capacity.)

3. In the BCI literature, subjects A01 and A03 are typically the strongest ("BCI literate") while A05 and A08 tend to be weaker. Do your results match this pattern? What does this suggest about whether the model or the signal quality is the bottleneck?

*Write your answers here.*

---

## Part 10 — Ablation Experiments

Now that you have a working baseline, let's understand which components matter.

### Ablation 1: Number of Transformer layers

The default uses `n_layers=6`. More layers means more self-attention steps and deeper feature processing, but also more parameters and higher overfitting risk.

**Hypothesis:** Write your prediction below before running the experiment.

> I predict that `n_layers=2` will ___ (improve / hurt / not change) accuracy because ___.
> I predict that `n_layers=10` will ___ because ___.

In [ ]:
# Ablation 1: Number of Transformer layers
# Test on Subject A01 only (for speed)

ABLATION1_PATH = 'metrics/conformer_ablation_layers.pt'

if os.path.exists(ABLATION1_PATH):
    ablation1_results = torch.load(ABLATION1_PATH)
    print("Loaded cached ablation results.")
else:
    layer_configs = [2, 4, 6, 8]
    ablation1_results = {}
    
    for n_layers in layer_configs:
        print(f"\nTraining with n_layers={n_layers}...")
        model = EEGConformer(n_layers=n_layers).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
        
        for epoch in range(N_EPOCHS):
            train_one_epoch(model, train_loader, optimizer, DEVICE)
            scheduler.step()
        
        acc, kappa, _, _ = evaluate(model, test_loader, DEVICE)
        n_params = sum(p.numel() for p in model.parameters())
        ablation1_results[n_layers] = {
            'accuracy': acc, 'kappa': kappa, 'n_params': n_params
        }
        print(f"  n_layers={n_layers}: acc={acc:.4f}, kappa={kappa:.4f}, params={n_params:,}")
    
    os.makedirs('metrics', exist_ok=True)
    torch.save(ablation1_results, ABLATION1_PATH)

# Display results
print(f"\n{'n_layers':<12} {'Accuracy':>10} {'Kappa':>10} {'Params':>12}")
print('-' * 46)
for n_layers in sorted(ablation1_results.keys()):
    r = ablation1_results[n_layers]
    print(f"{n_layers:<12} {r['accuracy']:>10.4f} {r['kappa']:>10.4f} {r['n_params']:>12,}")

**Observation — Number of layers:**

> The best number of layers was ___. Increasing from 6 to 8 ___ accuracy by ___ points.
> Decreasing to 2 layers ___ accuracy by ___ points.
> The parameter count grows ___ (linearly / quadratically) with layers because ___.
> For our dataset size (~288 trials), ___ layers appears optimal. This is consistent with the EEG literature because ___.

*Fill in.*

### Ablation 2: Number of attention heads

With `d_model=40`, the number of heads must divide 40 evenly. More heads = more diverse attention patterns but smaller per-head dimension.

**Hypothesis:**
> I predict that `n_heads=1` (a single attention head with d_k=40) will ___ compared to `n_heads=10` (d_k=4 each) because ___.

In [ ]:
# Ablation 2: Number of attention heads

ABLATION2_PATH = 'metrics/conformer_ablation_heads.pt'

if os.path.exists(ABLATION2_PATH):
    ablation2_results = torch.load(ABLATION2_PATH)
    print("Loaded cached ablation results.")
else:
    head_configs = [1, 2, 5, 8, 10]  # all divide 40
    ablation2_results = {}
    
    for n_heads in head_configs:
        print(f"\nTraining with n_heads={n_heads} (d_k={40//n_heads})...")
        model = EEGConformer(n_heads=n_heads).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
        
        for epoch in range(N_EPOCHS):
            train_one_epoch(model, train_loader, optimizer, DEVICE)
            scheduler.step()
        
        acc, kappa, _, _ = evaluate(model, test_loader, DEVICE)
        ablation2_results[n_heads] = {'accuracy': acc, 'kappa': kappa, 'd_k': 40 // n_heads}
        print(f"  n_heads={n_heads}: acc={acc:.4f}, kappa={kappa:.4f}")
    
    os.makedirs('metrics', exist_ok=True)
    torch.save(ablation2_results, ABLATION2_PATH)

# Display results
print(f"\n{'n_heads':<10} {'d_k':>6} {'Accuracy':>10} {'Kappa':>10}")
print('-' * 38)
for n_heads in sorted(ablation2_results.keys()):
    r = ablation2_results[n_heads]
    print(f"{n_heads:<10} {r['d_k']:>6} {r['accuracy']:>10.4f} {r['kappa']:>10.4f}")

**Observation — Attention heads:**

> The best number of heads was ___. A single head (d_k=40) performed ___ compared to multiple heads.
> This suggests that for our 12-token EEG sequences, the model ___ (benefits from / is indifferent to / is hurt by) multiple diverse attention patterns.
> In NLP Transformers with d_model=768 and sequences of 512+ tokens, n_heads=12 is standard. The difference here is ___.

*Fill in.*

### Ablation 3: CNN front-end kernel length

The temporal kernel in the CNN front-end determines what frequency content each patch captures locally. A longer kernel sees more temporal context but might overlap with the Transformer's global modeling.

**Hypothesis:**
> I predict that `kern_len=5` (~20ms, very short) will ___ because ___.
> I predict that `kern_len=65` (~260ms, approaching EEGNet's scale) will ___ because ___.

In [ ]:
# Ablation 3: Temporal kernel length

ABLATION3_PATH = 'metrics/conformer_ablation_kern.pt'

if os.path.exists(ABLATION3_PATH):
    ablation3_results = torch.load(ABLATION3_PATH)
    print("Loaded cached ablation results.")
else:
    kern_configs = [5, 15, 25, 45, 65]
    ablation3_results = {}
    
    for kern_len in kern_configs:
        print(f"\nTraining with kern_len={kern_len} ({kern_len/250*1000:.0f}ms)...")
        model = EEGConformer(kern_len=kern_len).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
        
        for epoch in range(N_EPOCHS):
            train_one_epoch(model, train_loader, optimizer, DEVICE)
            scheduler.step()
        
        acc, kappa, _, _ = evaluate(model, test_loader, DEVICE)
        ablation3_results[kern_len] = {
            'accuracy': acc, 'kappa': kappa,
            'duration_ms': kern_len / 250 * 1000
        }
        print(f"  kern_len={kern_len}: acc={acc:.4f}, kappa={kappa:.4f}")
    
    os.makedirs('metrics', exist_ok=True)
    torch.save(ablation3_results, ABLATION3_PATH)

# Display results
print(f"\n{'kern_len':<12} {'Duration':>10} {'Accuracy':>10} {'Kappa':>10}")
print('-' * 44)
for kern in sorted(ablation3_results.keys()):
    r = ablation3_results[kern]
    print(f"{kern:<12} {r['duration_ms']:>8.0f}ms {r['accuracy']:>10.4f} {r['kappa']:>10.4f}")

**Observation — Temporal kernel length:**

> The best kernel length was ___ (___ms). A very short kernel (5, ~20ms) performed ___ because ___.
> A long kernel (65, ~260ms) performed ___ because ___.
> The Conformer's CNN front-end optimal kernel length is ___ (shorter / longer / similar to) EEGNet's 125-sample kernel. This makes sense because ___.
> The key architectural insight: the CNN front-end in the Conformer does NOT need to capture the full frequency band of interest — it only needs to provide ___ for the Transformer to then ___.

*Fill in.*

---

## Part 11 — Visualizing Attention Patterns

### What attention tells us about temporal reasoning

One of the advantages of Transformers over CNNs is interpretability through attention weights. Each attention head produces a `(T', T')` matrix showing how much each token attends to every other token.

For motor imagery EEG, we'd expect to see:
- **Broad attention** in early layers — gathering global context across the full 1-second window.
- **Focused attention** in later layers — attending to the most discriminative time segments (e.g. the ERD onset period).
- **Head specialization** — different heads might attend to different temporal relationships (e.g. one head focuses on early-to-late, another on adjacent tokens).

We'll visualize attention for one correctly classified trial per class.

In [ ]:
# Load the trained A01 model
model_a01 = EEGConformer().to(DEVICE)
model_a01.load_state_dict(torch.load(MODEL_PATH_A01, map_location=DEVICE))
model_a01.eval()

# Get predictions for test set
_, _, preds_a01, labels_a01 = evaluate(model_a01, test_loader, DEVICE)

# Find one correctly classified example per class
class_names = ['Left Hand', 'Right Hand', 'Feet', 'Tongue']
examples = {}

for c in range(4):
    correct_mask = (preds_a01 == c) & (labels_a01 == c)
    if correct_mask.any():
        idx = np.where(correct_mask)[0][0]
        x_sample = torch.tensor(X_test[idx:idx+1], dtype=torch.float32).unsqueeze(1).to(DEVICE)
        attn_list = model_a01.get_attention_weights(x_sample)
        examples[c] = attn_list
        print(f"Class {c} ({class_names[c]}): using test sample {idx}")

# Plot attention matrices from the last layer for each class
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for c in range(4):
    if c in examples:
        # Last layer, averaged across heads, single sample
        attn = examples[c][-1][0].numpy()  # (T', T')
        im = axes[c].imshow(attn, cmap='viridis', aspect='equal')
        axes[c].set_title(f'{class_names[c]}')
        axes[c].set_xlabel('Key position')
        if c == 0:
            axes[c].set_ylabel('Query position')
        plt.colorbar(im, ax=axes[c], fraction=0.046, pad=0.04)

fig.suptitle('Attention Patterns (Last Layer, Avg over Heads) — A01', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Compare attention across layers for one class (e.g. Left Hand)
if 0 in examples:
    n_layers_vis = len(examples[0])
    fig, axes = plt.subplots(1, n_layers_vis, figsize=(3 * n_layers_vis, 3))
    
    for layer_idx in range(n_layers_vis):
        attn = examples[0][layer_idx][0].numpy()  # (T', T')
        im = axes[layer_idx].imshow(attn, cmap='viridis', aspect='equal')
        axes[layer_idx].set_title(f'Layer {layer_idx + 1}')
        axes[layer_idx].set_xlabel('Key')
        if layer_idx == 0:
            axes[layer_idx].set_ylabel('Query')
    
    fig.suptitle('Attention Across Layers — Left Hand (A01)', fontsize=12)
    plt.tight_layout()
    plt.show()

### Decision — Attention patterns

1. Do the attention patterns differ between classes? If Left Hand and Right Hand show different attention distributions, what might this tell us about how the Conformer discriminates between the two? (Think about: which temporal segments are most attended to, and what EEG events happen in those time ranges.)

2. Compare Layer 1 attention to Layer 6 attention. Do you see the expected pattern of broad → focused attention, or something else? If the attention is relatively uniform (near-flat rows), what does that mean — is the model not using attention, or is attention simply distributing information evenly?

3. If attention patterns look nearly identical across all four classes, does that mean the Transformer is useless? Or might the class-discriminative information be encoded in the *values* rather than the *attention weights*? (Hint: attention weights control *where* to look; the value vectors control *what* to extract.)

*Write your answers here.*

---

## Part 12 — Model Comparison Summary

Create a comprehensive comparison of all three models for the capstone.

In [ ]:
# Save final summary for capstone comparison
summary = {
    'method': 'EEG Conformer',
    'per_subject': all_results,
    'mean_accuracy': float(np.mean(accs)),
    'std_accuracy': float(np.std(accs)),
    'mean_kappa': float(np.mean(kappas)),
    'std_kappa': float(np.std(kappas)),
    'hyperparams': {
        'F1': 40, 'D': 1, 'kern_len': 25,
        'pool_size': 75, 'pool_stride': 15,
        'n_heads': 10, 'n_layers': 6, 'ff_ratio': 3,
        'dropout': 0.5,
        'lr': LR, 'epochs': N_EPOCHS, 'batch_size': BATCH_SIZE,
        'weight_decay': WEIGHT_DECAY,
        'optimizer': 'AdamW', 'scheduler': 'CosineAnnealingLR'
    }
}

torch.save(summary, 'metrics/conformer_summary.pt')
print("EEG Conformer summary saved for capstone comparison.")
print(f"\nFinal results:")
print(f"  Accuracy: {summary['mean_accuracy']:.3f} ± {summary['std_accuracy']:.3f}")
print(f"  Kappa:    {summary['mean_kappa']:.3f} ± {summary['std_kappa']:.3f}")

In [ ]:
# Quick comparison table (fill in your CSP and EEGNet numbers)
print("\n" + "="*60)
print("MODEL COMPARISON — BCI Competition IV Dataset 2a")
print("="*60)
print(f"{'Model':<20} {'Accuracy':>12} {'Kappa':>12}")
print("-"*46)
print(f"{'CSP + SVM':<20} {'___±___':>12} {'___±___':>12}")  # fill from your CSP results
print(f"{'EEGNet':<20} {'___±___':>12} {'___±___':>12}")     # fill from your EEGNet results
print(f"{'EEG Conformer':<20} {summary['mean_accuracy']:.3f}±{summary['std_accuracy']:.3f:>7} "
      f"{summary['mean_kappa']:.3f}±{summary['std_kappa']:.3f:>7}")
print("-"*46)
print("\n(Fill in CSP and EEGNet numbers from your previous notebooks)")

---

## Summary

In this notebook you:

1. Learned **patch tokenization for EEG** — a CNN front-end (temporal + depthwise spatial convolution) that converts raw `(1, 22, 250)` EEG into a sequence of `(T', d_model)` tokens. Each token represents a compressed temporal patch of the multi-channel signal.

2. Implemented the **EEG Conformer** — a hybrid CNN-Transformer architecture where the CNN handles local spatial-temporal feature extraction and the Transformer models global temporal dependencies via self-attention.

3. Compared the Conformer to both EEGNet and CSP+SVM across all 9 subjects, observing how architectural complexity trades off with generalization on small EEG datasets (~288 trials).

4. Conducted ablation studies on the number of Transformer layers, attention heads, and CNN kernel length — revealing the relative contribution of each component.

5. Visualized **attention patterns** to understand what temporal relationships the Conformer learns for different motor imagery classes.

### Key takeaways for BCI engineering

- **CNNs are not enough for long-range temporal reasoning.** EEGNet's fixed-kernel architecture misses dependencies beyond its receptive field. The Transformer adds global context.
- **Transformers alone are not enough for EEG.** Raw EEG patches are too noisy for attention to work directly — the CNN front-end is essential for denoising and feature extraction.
- **Small data limits large models.** With ~288 trials, the Conformer's advantage over EEGNet may be modest or even negative for some subjects. Heavy dropout, weight decay, and conservative model sizing are critical.
- **Attention interpretability has limits.** Uniform attention doesn't mean the Transformer is useless — the information may be carried in the value vectors rather than the attention pattern.

### What's next

In the **capstone notebook** (Notebook 7), you'll:
- Run all three models (CSP, EEGNet, Conformer) under identical evaluation conditions
- Compare subject-dependent vs subject-independent paradigms
- Use saliency maps / input gradients for model explainability
- Produce the final results table and figures for your project report